In [2]:
import torch
import sys
sys.path.append("..")
from modules.model import TumorClassifier
from modules.transforms import train_transform, val_transform, test_transform
import onnx

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TumorClassifier()
model = model.to(device)
model.load_state_dict(torch.load("../training_data/effnet_weights_sampler_cosine_best_model.pth"))
model.eval()  # Set the model to evaluation mode

In [8]:
# Prepare dummy input
dummy_input = torch.randn(1, 3, 224, 224).to(device)

# Export
torch.onnx.export(
    model,
    dummy_input,
    "brain_tumor_classifier.onnx",
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

# Optional: Validate the ONNX model
onnx_model = onnx.load("brain_tumor_classifier.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model is valid!")

ONNX model is valid!


In [1]:
import tensorrt as trt

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

def build_engine(onnx_file_path, engine_file_path):
    builder = trt.Builder(TRT_LOGGER)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, TRT_LOGGER)

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1 GB

    if builder.platform_has_fast_fp16:
        config.set_flag(trt.BuilderFlag.FP16)
    else:
        print("Warning: Platform does not support fast FP16. Falling back to FP32.")

    with open(onnx_file_path, 'rb') as model:
        if not parser.parse(model.read()):
            print('Failed to parse the ONNX file.')
            for error in range(parser.num_errors):
                print(parser.get_error(error))
            return None

    # Create an optimization profile (for dynamic shapes)
    profile = builder.create_optimization_profile()
    input_name = network.get_input(0).name

    # For example: (batch_size, channels, height, width)
    profile.set_shape(input_name, (1, 3, 224, 224), (1, 3, 224, 224), (1, 3, 224, 224))
    config.add_optimization_profile(profile)

    engine = builder.build_serialized_network(network, config)
    with open(engine_file_path, 'wb') as f:
        f.write(engine)  # Directly write

    print("TensorRT engine created successfully!")

build_engine("brain_tumor_classifier.onnx", "brain_tumor.engine")


TensorRT engine created successfully!
